# Autoencoder and Variational Autoencoder Live Demo
This notebook contains the complete pipeline for data processing, model definitions, and training for the Autoencoder and Variational Autoencoder.

## 0. Setup Colab Environment
The following cell mounts Google Drive and downloads the Medical MNIST dataset using kagglehub.

In [ ]:
!pip install kagglehub
from google.colab import drive
import os
import shutil
import kagglehub

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Download dataset from Kaggle
print("Downloading dataset from Kaggle...")
os.environ["KAGGLEHUB_VERIFY_SSL"] = "0"
temp_path = kagglehub.dataset_download("andrewmvd/medical-mnist")
print("Downloaded to temporary path:", temp_path)

# 3. Define the destination path in Google Drive
DRIVE_DATA_DIR = '/content/drive/MyDrive/medical-mnist'

# 4. Upload/Move dataset to Google Drive if it doesn't exist
if not os.path.exists(DRIVE_DATA_DIR):
    print(f"Moving dataset to Google Drive: {DRIVE_DATA_DIR}...")
    shutil.copytree(temp_path, DRIVE_DATA_DIR)
    print("Dataset successfully uploaded to Google Drive.")
else:
    print(f"Dataset already exists in Google Drive at {DRIVE_DATA_DIR}.")


## 1. Data Processing
The following cell defines how the dataset is loaded and preprocessed.

In [ ]:
import tensorflow as tf

def get_dataset(data_dir, batch_size=32, image_size=(64, 64), validation_split=0.2, seed=123):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        labels=None, # Unsupervised learning
        color_mode='grayscale',
        batch_size=batch_size,
        image_size=image_size,
        validation_split=validation_split,
        subset="training",
        seed=seed,
    )
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        labels=None,
        color_mode='grayscale',
        batch_size=batch_size,
        image_size=image_size,
        validation_split=validation_split,
        subset="validation",
        seed=seed,
    )

    normalization_layer = tf.keras.layers.Rescaling(1./255)
    train_ds = train_ds.map(lambda x: (normalization_layer(x), normalization_layer(x)))
    val_ds = val_ds.map(lambda x: (normalization_layer(x), normalization_layer(x)))

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

    return train_ds, val_ds

def add_noise(images, noise_factor=0.2):
    noise = tf.random.normal(shape=tf.shape(images), mean=0.0, stddev=noise_factor, dtype=tf.float32)
    noisy_images = images + noise
    return tf.clip_by_value(noisy_images, clip_value_min=0., clip_value_max=1.)

def get_noisy_dataset(dataset, noise_factor=0.2):
    return dataset.map(lambda x, y: (add_noise(x, noise_factor), y))


## 2. Models
The following cell defines the Autoencoder and Variational Autoencoder architectures.

In [ ]:
class Autoencoder(tf.keras.Model):
    def __init__(self, latent_dim=64, input_shape=(64, 64, 1)):
        super(Autoencoder, self).__init__()
        self.latent_dim = latent_dim
        
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=input_shape),
            tf.keras.layers.Conv2D(32, 3, activation='relu', strides=2, padding='same'),
            tf.keras.layers.Conv2D(64, 3, activation='relu', strides=2, padding='same'),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(latent_dim)
        ], name='ae_encoder')

        self.decoder = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(latent_dim,)),
            tf.keras.layers.Dense(16 * 16 * 64, activation='relu'),
            tf.keras.layers.Reshape((16, 16, 64)),
            tf.keras.layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same'),
            tf.keras.layers.Conv2DTranspose(32, 3, activation='relu', strides=2, padding='same'),
            tf.keras.layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')
        ], name='ae_decoder')

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

class Sampling(tf.keras.layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

class VariationalAutoencoder(tf.keras.Model):
    def __init__(self, latent_dim=64, input_shape=(64, 64, 1)):
        super(VariationalAutoencoder, self).__init__()
        self.latent_dim = latent_dim

        encoder_inputs = tf.keras.Input(shape=input_shape)
        x = tf.keras.layers.Conv2D(32, 3, activation="relu", strides=2, padding="same")(encoder_inputs)
        x = tf.keras.layers.Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(128, activation="relu")(x)
        z_mean = tf.keras.layers.Dense(latent_dim, name="z_mean")(x)
        z_log_var = tf.keras.layers.Dense(latent_dim, name="z_log_var")(x)
        z = Sampling()([z_mean, z_log_var])
        self.encoder = tf.keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="vae_encoder")

        latent_inputs = tf.keras.Input(shape=(latent_dim,))
        x = tf.keras.layers.Dense(16 * 16 * 64, activation="relu")(latent_inputs)
        x = tf.keras.layers.Reshape((16, 16, 64))(x)
        x = tf.keras.layers.Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
        x = tf.keras.layers.Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
        decoder_outputs = tf.keras.layers.Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(x)
        self.decoder = tf.keras.Model(latent_inputs, decoder_outputs, name="vae_decoder")

        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        return reconstruction

    def train_step(self, data):
        if isinstance(data, tuple):
            x = data[0]
        else:
            x = data
            
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x)
            reconstruction = self.decoder(z)
            
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.keras.losses.mean_squared_error(x, reconstruction), axis=(1, 2)
                )
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
            
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        if isinstance(data, tuple):
            x = data[0]
        else:
            x = data

        z_mean, z_log_var, z = self.encoder(x)
        reconstruction = self.decoder(z)
        
        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(
                tf.keras.losses.mean_squared_error(x, reconstruction), axis=(1, 2)
            )
        )
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
        total_loss = reconstruction_loss + kl_loss

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }


## 3. Training
The following cell trains the models and saves the checkpoints. Ensure your data is extracted in `../data/raw/` relative to this notebook.

In [ ]:
import os

data_dir = '/content/drive/MyDrive/medical-mnist'
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

print("Loading dataset...")
train_ds, val_ds = get_dataset(data_dir, batch_size=32)

# Train Autoencoder
print("Training Autoencoder...")
ae = Autoencoder()
ae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse')

ae_history = ae.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds
)

# Train Variational Autoencoder
print("Training Variational Autoencoder...")
vae = VariationalAutoencoder()
vae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3))

vae_history = vae.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds
)
